# Vision-IA · entrenamiento A/B con Drive sincronizado

Usa siempre este notebook. En Windows ejecuta `python -m scripts.prepare_training --experiment B` (o A). El paquete y la selección se publican automáticamente en tu carpeta de Drive. Deja `python -m scripts.sync_results --watch` para recibir resultados sin descargar ni copiar ZIPs.

No se automatiza el inicio de sesión ni se entra a Colab desde Windows. Tú autorizas Drive y ejecutas la celda de entrenamiento. Test permanece reservado y cada entrenamiento inicia la misma CNN desde cero.


## 1 · Autoriza Drive y comprueba la GPU

Selecciona GPU en Colab (T4 si está disponible). Marca la casilla cuando quieras autorizar el montaje y ejecuta. `RUTA_COMPARTIDA` es relativa a Mi unidad, no una ruta de Windows. El valor actual corresponde a Mi unidad/Vision-IA.


In [ ]:
AUTORIZAR_MONTAJE = False #@param {type:"boolean"}
RUTA_COMPARTIDA = 'Vision-IA' #@param {type:"string"}
EXPERIMENTO = 'desde_control' #@param ['desde_control', 'A', 'B']
SETUP_READY = False
if not AUTORIZAR_MONTAJE:
    raise RuntimeError('Autoriza Drive marcando la casilla y vuelve a ejecutar esta celda.')
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from pathlib import PurePosixPath
import torch
if PurePosixPath(RUTA_COMPARTIDA).is_absolute() or '..' in PurePosixPath(RUTA_COMPARTIDA).parts or '\\' in RUTA_COMPARTIDA:
    raise ValueError('Usa una ruta relativa a Mi unidad.')
SHARED_DIR = Path('/content/drive/MyDrive') / RUTA_COMPARTIDA
if not SHARED_DIR.is_dir():
    raise FileNotFoundError('No existe la carpeta compartida. Revisa RUTA_COMPARTIDA y la cuenta de Google.')
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU asignada:', torch.cuda.get_device_name(0))
else:
    print('Activa GPU antes de iniciar entrenamiento. Las siguientes verificaciones no entrenan.')


## 2 · Obtener el experimento preparado, verificar e instalar

Deja `EXPERIMENTO = desde_control` para usar la última selección publicada desde Windows. Este paso lee `control.json`, comprueba tamaño/SHA-256, extrae una copia local nueva y verifica datos train/val. Si Drive aún sincroniza, espera a que termine y repite esta celda. No subas ZIPs manualmente.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

CONTROL = json.loads((SHARED_DIR / 'control.json').read_text(encoding='utf-8'))
SELECTED = CONTROL['selected_experiment'] if EXPERIMENTO == 'desde_control' else EXPERIMENTO
if SELECTED not in ('A', 'B'):
    raise ValueError('El experimento debe ser A o B.')
if SELECTED not in CONTROL['packages']:
    raise FileNotFoundError('Primero prepara ese experimento desde Windows con scripts.prepare_training.')
ENTRY = CONTROL['packages'][SELECTED]
if ENTRY['experiment'] != SELECTED:
    raise ValueError('La selección y el paquete no coinciden.')
relative = ENTRY['package_relative_path']
PACKAGE_SOURCE = (SHARED_DIR / relative).resolve()
if not PACKAGE_SOURCE.is_relative_to(SHARED_DIR.resolve()) or '..' in PurePosixPath(relative).parts:
    raise ValueError('Ruta de paquete fuera de la carpeta compartida.')
if not PACKAGE_SOURCE.is_file() or PACKAGE_SOURCE.stat().st_size != ENTRY['bytes']:
    raise RuntimeError('El paquete aún no está sincronizado: espera a Drive y reintenta.')
WORKSPACE = Path(tempfile.mkdtemp(prefix='vision_ia_drive_', dir='/content'))
LOCAL_ZIP = WORKSPACE / ('Vision-IA_Colab_' + SELECTED + '.zip')
shutil.copyfile(PACKAGE_SOURCE, LOCAL_ZIP)
if file_hash(LOCAL_ZIP) != ENTRY['package_sha256']:
    raise RuntimeError('SHA-256 distinto: Drive aún sincroniza o el paquete está alterado. No se entrenará.')
with zipfile.ZipFile(LOCAL_ZIP) as bundle:
    for member in bundle.infolist():
        parts = PurePosixPath(member.filename).parts
        if not parts or parts[0] != 'Vision-IA' or '..' in parts or '\\' in member.filename or PurePosixPath(member.filename).is_absolute():
            raise ValueError('Ruta ZIP insegura.')
        if ((member.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('No se permiten enlaces simbólicos.')
    bundle.extractall(WORKSPACE)
PROJECT_DIR = WORKSPACE / 'Vision-IA'
index = json.loads((PROJECT_DIR / 'PACKAGE_MANIFEST.json').read_text(encoding='utf-8'))
if index['experiment'] != SELECTED or index.get('workflow') != 'drive':
    raise ValueError('Paquete de otro experimento o flujo.')
for item in index['files']:
    path = (PROJECT_DIR / item['path']).resolve()
    if not path.is_relative_to(PROJECT_DIR.resolve()) or not path.is_file() or path.stat().st_size != item['bytes'] or file_hash(path) != item['sha256']:
        raise ValueError('Archivo del paquete incompleto: ' + item['path'])
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], cwd=PROJECT_DIR, check=True)
CONFIG = 'config/train_' + SELECTED + '.json'
subprocess.run([sys.executable, '-m', 'scripts.inspect_training_data', '--config', CONFIG, '--verify-hashes'], cwd=PROJECT_DIR, check=True)
SETUP_READY = True
print('Listo:', SELECTED, '| imágenes:', index['image_count'], '| paquete:', ENTRY['package_id'])


## 3 · Revisar el plan, sin entrenar

Los parámetros iniciales están en la configuración: hasta 30 épocas, batch 32, learning rate 0.001. Si necesitas cambiarlos, edita `training_overrides` en config/sync.json en Windows y vuelve a preparar, sin reconstruir los datos. Cada ejecución tendrá su propia carpeta en `results/A/` o `results/B/`.


In [ ]:
if not SETUP_READY:
    raise RuntimeError('Primero completa la preparación de la celda 2.')
OVERRIDES = CONTROL.get('training_overrides', {})
allowed = {'epochs', 'batch_size', 'learning_rate', 'num_workers'}
if set(OVERRIDES) - allowed:
    raise ValueError('Overrides no permitidos: no se admiten checkpoints de inicialización.')
RECEIPT = WORKSPACE / 'run_receipt.json'
RUN_COMMAND = [sys.executable, '-m', 'scripts.run_colab', '--experiment', SELECTED,
               '--results-root', str(SHARED_DIR / 'results'), '--package-id', ENTRY['package_id'],
               '--device', 'cuda', '--receipt', str(RECEIPT)]
for key, value in OVERRIDES.items():
    RUN_COMMAND += ['--' + key.replace('_', '-'), str(value)]
subprocess.run(RUN_COMMAND + ['--dry-run'], cwd=PROJECT_DIR, check=True)
print('Plan provisional verificado; todavía no se ha iniciado entrenamiento.')


## 4 · Entrenar — esta celda inicia el entrenamiento real

Ejecuta cuando estés listo. Se crea la misma VisionCNN-v1 de 89.686 parámetros con pesos aleatorios. No se carga el modelo A para entrenar B. Modelos, historial, curvas y registros se guardan en Drive durante la ejecución; `FINISHED.json` se escribe solo al finalizar. No se usa test.


In [ ]:
if not SETUP_READY:
    raise RuntimeError('Primero prepara el paquete.')
if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU CUDA; selecciona un entorno GPU antes de entrenar.')
subprocess.run(RUN_COMMAND, cwd=PROJECT_DIR, check=True)
RUN_RECEIPT = json.loads(RECEIPT.read_text(encoding='utf-8'))
if RUN_RECEIPT['experiment'] != SELECTED or RUN_RECEIPT['package_id'] != ENTRY['package_id']:
    raise ValueError('La ejecución no corresponde al paquete elegido.')
RUN_DIR = Path(RUN_RECEIPT['output'])
print('Resultados guardados automáticamente:', RUN_DIR)


## 5 · Ver los resultados reales

Windows los recibirá mediante Drive para escritorio. Deja ejecutándose `scripts.sync_results --watch` o ejecútalo una vez al terminar. No hace falta descargar ni copiar el modelo a mano; se conserva en `runs/synced/<experimento>/<ejecución>/` sin sobrescribir models/best_model.pth de A.


In [ ]:
from IPython.display import display, Image as DisplayImage
summary = json.loads((RUN_DIR / 'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
display(DisplayImage(filename=str(RUN_DIR / 'training_curves.png')))
print('Mejor modelo:', RUN_DIR / 'best_model.pth')
print('Marcador para sincronización:', RUN_DIR / 'FINISHED.json')


## Qué sigue siendo manual

Colab requiere que selecciones GPU, autorices Drive e inicies la celda de entrenamiento. No se automatiza autenticación, disponibilidad de GPU o permanencia de la sesión. Si se interrumpe antes de finalizar, Windows espera o muestra el fallo; no importa un modelo parcial como entrenamiento completo. No se reanuda Adam automáticamente: una ejecución nueva inicia de cero y usa otra carpeta.

A y B tienen particiones diferentes. Sus métricas de validación sirven para analizar cada ejecución, pero no sustituyen una evaluación independiente común. Test permanece reservado. La clase orgánica sigue procediendo de RealWaste; conservar la advertencia sobre sesgo y las atribuciones de docs/DATASETS.md.
